In [ ]:
from concurrent.futures import ThreadPoolExecutor

import geopandas as gpd
import rasterio as rio

In [10]:
DATA_PREFIX = "../data"
OUTPUT_PREFIX = "../output"

TRAIN_PARQUET = f"{OUTPUT_PREFIX}/train_only_biomass.parquet"
TEST_PARQUET = f"{OUTPUT_PREFIX}/test_only_biomass.parquet"
EXTRACT_PARQUET = f"{OUTPUT_PREFIX}/extract_train.parquet"
SUBMISSION_CSV = f"{DATA_PREFIX}/sample_submission.csv"

In [6]:
BANDS_S2 = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
BANDS_S1 = ["VV", "VH"]

INDICES = [
    dict(name="NDVI", band1="NIR", band2="RED"),
    dict(name="NDMI", band1="NIR", band2="SWIR1"),
    dict(name="NBR", band1="NIR", band2="SWIR2"),
    dict(name="NBR2", band1="SWIR1", band2="SWIR2"),
    dict(name="NDWI", band1="GREEN", band2="NIR"),
    dict(name="MNDWI", band1="GREEN", band2="SWIR1"),
    dict(name="MNDWI2", band1="GREEN", band2="SWIR2"),
    dict(name="RVI", band1="VV", band2="VH"),
]

INDICES_BANDS = [indi["name"] for indi in INDICES]

PREDICTORS = [*BANDS_S2, *BANDS_S1, *INDICES_BANDS]

LABEL = "biomass"

In [5]:
train_df = gpd.read_parquet(TRAIN_PARQUET)
train_df

,x,y,year,biomass,tile_id,geometry
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217)
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357)
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368)
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331)
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668)
...,...,...,...,...,...,...
5165029,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312)
5165030,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261)
5165031,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345)
5165032,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471)


In [8]:
tile_ids = train_df["tile_id"].unique()


def run_per_tile(index):
    tile_id = tile_ids[index]
    tile_sample = train_df[train_df["tile_id"] == tile_id]
    coords = [coord for coord in zip(tile_sample.geometry.x, tile_sample.geometry.y)]
    years = tile_sample["year"].unique()

    for year in years:
        print(f"Run S2 {tile_id} {year} {index + 1} / {len(tile_ids)}")

        with rio.open(
            f"https://storage.googleapis.com/gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s2/tile_{tile_id}_{year}_S2_L2A_composite_{year}-06-01_{year}-07-31_10m.tif"
        ) as src:
            train_df.loc[
                (train_df["tile_id"] == tile_id) & (train_df["year"] == year), BANDS_S2
            ] = [data for data in src.sample(coords)]

        print(f"Run S1 {tile_id} {year} {index + 1} / {len(tile_ids)}")

        with rio.open(
            f"https://storage.googleapis.com/gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s1/tile_{tile_id}_{year}_S1_RTC_composite_{year}-06-01_{year}-07-31_10m.tif"
        ) as src:
            train_df.loc[
                (train_df["tile_id"] == tile_id) & (train_df["year"] == year), BANDS_S1
            ] = [data for data in src.sample(coords)]


with ThreadPoolExecutor(8) as executor:
    jobs = []
    for index in range(len(tile_ids)):
        jobs.append(executor.submit(run_per_tile, index))
    for job in jobs:
        try:
            job.result()
        except Exception as e:
            print(f"Error: {e}")

train_df

Run S2 030430 2019Run S2 036385 2019

Run S2 038052 2020
Run S2 026261 2021
Run S2 086895 2016
Run S2 039974 2019
Run S2 038510 2019
Run S2 051927 2019
Run S1 030430 2019
Run S1 026261 2021
Run S2 045997 2019
Run S2 088988 2016
Run S1 036385 2019
Run S2 085676 2016
Run S1 085676 2016
Run S2 034493 2019
Run S1 039974 2019
Run S1 045997 2019
Run S1 086895 2016
Run S1 051927 2019
Run S1 038510 2019
Run S1 038052 2020
Run S2 040151 2019
Run S1 034493 2019
Run S2 083291 2016
Run S2 029528 2019
Run S1 029528 2019
Run S2 034594 2019
Run S1 034594 2019
Run S2 042237 2019
Run S2 045356 2019
Run S2 045312 2019
Run S2 026863 2021
Run S1 045312 2019
Run S2 041342 2019
Run S1 083291 2016Run S2 092872 2016

Run S1 088988 2016
Run S1 040151 2019
Run S2 041182 2019
Run S1 042237 2019
Run S1 041342 2019
Run S1 045356 2019
Run S1 041182 2019
Run S2 098252 2016
Run S2 051637 2019
Run S2 040307 2019
Run S1 026863 2021
Run S2 061154 2020
Run S1 061154 2020
Run S2 045647 2019
Run S2 044463 2019
Run S2 02716

/home/ramiqcom/application/opengeohub-summerschool-2026/.venv/lib/python3.11/site-packages/pyogrio/raw.py:733: RuntimeWarning: 2GB file size limit reached for ../output/extract_train.parquet/extract_train.dbf. Going on, but might cause compatibility issues with third party software
  ogr_write(


,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,SWIR1,SWIR2,VV,VH
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,2102.0,1286.0,1085.0,303.0
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,2099.0,1065.0,1721.0,465.0
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,2441.0,1518.0,1105.0,346.0
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,2593.0,1707.0,995.0,356.0
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,1485.0,878.0,929.0,383.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5165029,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,2694.0,1714.0,1295.0,304.0
5165030,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261),353.0,593.0,527.0,3129.0,2419.0,1430.0,1396.0,278.0
5165031,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,2165.0,1215.0,1043.0,375.0
5165032,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,1958.0,1190.0,1443.0,439.0


In [11]:
# to parquet
train_df.to_parquet(EXTRACT_PARQUET)

In [12]:
test_df = gpd.read_parquet(TEST_PARQUET)
test_df

,row_id,tile_id,x,y,year,geometry
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662)
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768)
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436)
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389)
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073)
...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472)
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492)
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469)
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512)


In [17]:
tile_ids = test_df["tile_id"].unique()


def run_per_tile(index):
    try:
        tile_id = tile_ids[index]
        tile_sample = test_df[test_df["tile_id"] == tile_id]
        coords = [
            coord for coord in zip(tile_sample.geometry.x, tile_sample.geometry.y)
        ]
        years = tile_sample["year"].unique()

        for year in years:
            print(f"Run S2 {tile_id} {year} {index + 1} / {len(tile_ids)}")

            with rio.open(
                f"https://storage.googleapis.com/gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s2/tile_{tile_id}_{year}_S2_L2A_composite_{year}-06-01_{year}-07-31_10m.tif"
            ) as src:
                test_df.loc[
                    (test_df["tile_id"] == tile_id) & (test_df["year"] == year),
                    BANDS_S2,
                ] = [data for data in src.sample(coords)]

            print(f"Run S1 {tile_id} {year} {index + 1} / {len(tile_ids)}")

            with rio.open(
                f"https://storage.googleapis.com/gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s1/tile_{tile_id}_{year}_S1_RTC_composite_{year}-06-01_{year}-07-31_10m.tif"
            ) as src:
                test_df.loc[
                    (test_df["tile_id"] == tile_id) & (test_df["year"] == year),
                    BANDS_S1,
                ] = [data for data in src.sample(coords)]
    except Exception as e:
        raise Exception(f"Error: {tile_id} {year} {e} {index + 1} / {len(tile_ids)}")


with ThreadPoolExecutor(8) as executor:
    jobs = []
    for index in range(len(tile_ids)):
        jobs.append(executor.submit(run_per_tile, index))
    for job in jobs:
        try:
            job.result()
        except Exception as e:
            print(f"Error: {e}")

test_df

Run S2 031028 2019 6 / 10
Run S2 035814 2019 4 / 10
Run S2 060543 2020 7 / 10
Run S2 061751 2020 9 / 10
Run S2 045101 2019 2 / 10
Run S2 047474 2019 8 / 10
Run S2 062333 2020 3 / 10
Run S2 040897 2019 10 / 10
Run S2 062332 2020 5 / 10
Run S2 028631 2021 1 / 10
Run S1 028631 2021 1 / 10
Error: Error: 045101 2019 HTTP response code: 404 2 / 10
Error: Error: 062333 2020 HTTP response code: 404 3 / 10
Error: Error: 035814 2019 HTTP response code: 404 4 / 10
Error: Error: 062332 2020 HTTP response code: 404 5 / 10
Error: Error: 031028 2019 HTTP response code: 404 6 / 10
Error: Error: 060543 2020 HTTP response code: 404 7 / 10
Error: Error: 047474 2019 HTTP response code: 404 8 / 10
Error: Error: 061751 2020 HTTP response code: 404 9 / 10
Error: Error: 040897 2019 HTTP response code: 404 10 / 10


,row_id,tile_id,x,y,year,geometry,BLUE,GREEN,RED,NIR,SWIR1,SWIR2,VV,VH
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662),147.0,318.0,152.0,3368.0,1293.0,561.0,2425.0,548.0
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768),155.0,318.0,167.0,3011.0,1256.0,539.0,1591.0,403.0
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436),159.0,330.0,148.0,4007.0,1171.0,486.0,1360.0,447.0
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389),163.0,329.0,167.0,3221.0,1318.0,559.0,1893.0,504.0
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073),139.0,276.0,123.0,4183.0,1180.0,431.0,1943.0,350.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
